# Lucas Phase 1: Full-Map Runnable Starter

## Goal
Build a portfolio-grade Phase 1 notebook that:
- runs a strong **two-way FE + robustness** core now,
- keeps **IV, dynamic panel, and forecasting** in the same workflow as explicit scaffolds,
- enforces a practical decision filter: keep the project only if it remains high-signal and feasible in <= 14 days.

## Hard Filter
Continue only if both hold:
1. Strong portfolio signal for applied economist / economic data analyst roles.
2. Realistic delivery timeline with stable execution.


## Phase 1 Execution Plan (Plan-First)

### What runs now (with current files)
- Data contract checks on `macro_growth_merged.csv`
- Two-way FE baseline for:
  - `inflation ~ m2_growth`
  - `gdp_growth ~ m2_growth`
- Robustness pack:
  - winsorization,
  - period split,
  - leave-one-country-out influence diagnostics.

### What is scaffold-only (unless extra files are added)
- IV / 2SLS branch (requires `data/phase1_instruments.csv`)
- FE + controls extension (requires `data/phase1_controls.csv`)
- Dynamic panel and forecasting extension (template + guarded execution)

### Stop/Continue Triggers
- Continue if FE sign/magnitude are stable and at least 2 robustness checks are coherent.
- Stop/pivot if data merges for controls/instruments become multiday bottlenecks or identification quality is not defensible.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path('/Users/stevenchung/Desktop/P12B_File/New project_')
BASE_DATA_PATH = PROJECT_ROOT / 'macro_growth_merged.csv'
CONTROLS_PATH = PROJECT_ROOT / 'data/phase1_controls.csv'
IV_PATH = PROJECT_ROOT / 'data/phase1_instruments.csv'

OUTPUT_ROOT = PROJECT_ROOT / 'outputs/phase1'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIG_DIR = OUTPUT_ROOT / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FAST_MODE = True
LOCO_MAX_COUNTRIES = 40 if FAST_MODE else None
WINSOR_LOWER = 0.01
WINSOR_UPPER = 0.99

EXPORT_TABLES = {}
EXPORT_FIGS = {}
LOG_LINES = []
FE_RESULTS = {}


def log(msg: str):
    print(msg)
    LOG_LINES.append(msg)


def winsorize_series(s: pd.Series, lower_q: float = 0.01, upper_q: float = 0.99) -> pd.Series:
    lo, hi = s.quantile(lower_q), s.quantile(upper_q)
    return s.clip(lower=lo, upper=hi)


def fit_twfe(df_in: pd.DataFrame, outcome: str, regressor: str = 'm2_growth'):
    work = df_in[['Country Name', 'year', outcome, regressor]].dropna().copy()
    work['year'] = work['year'].astype(int)
    panel = work.set_index(['Country Name', 'year']).sort_index()
    model = PanelOLS.from_formula(
        f'{outcome} ~ 1 + {regressor} + EntityEffects + TimeEffects',
        data=panel,
    )
    result = model.fit(cov_type='clustered', cluster_entity=True)
    return result, panel


def result_to_table(result, model_name: str, outcome: str) -> pd.DataFrame:
    rows = []
    for term in result.params.index:
        rows.append(
            {
                'model_name': model_name,
                'outcome': outcome,
                'term': term,
                'coef': float(result.params.loc[term]),
                'std_error': float(result.std_errors.loc[term]),
                'p_value': float(result.pvalues.loc[term]),
                'nobs': int(result.nobs),
                'r2_within': float(result.rsquared_within),
                'r2_between': float(result.rsquared_between),
                'r2_overall': float(result.rsquared_overall),
            }
        )
    return pd.DataFrame(rows)


log('Environment initialized.')
log(f'Base data path: {BASE_DATA_PATH}')
log(f'Controls path (optional): {CONTROLS_PATH}')
log(f'IV path (optional): {IV_PATH}')
log(f'FAST_MODE={FAST_MODE}, LOCO_MAX_COUNTRIES={LOCO_MAX_COUNTRIES}')


In [ ]:
required_cols = ['Country Name', 'year', 'm2_growth', 'inflation', 'gdp_growth']

if not BASE_DATA_PATH.exists():
    raise FileNotFoundError(f'Missing base file: {BASE_DATA_PATH}')

df_base = pd.read_csv(BASE_DATA_PATH)
missing_cols = [c for c in required_cols if c not in df_base.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')
if df_base.empty:
    raise ValueError('Base dataset is empty.')

# Enforce expected dtypes where possible.
df_base['Country Name'] = df_base['Country Name'].astype(str)
df_base['year'] = pd.to_numeric(df_base['year'], errors='coerce').astype('Int64')
for c in ['m2_growth', 'inflation', 'gdp_growth']:
    df_base[c] = pd.to_numeric(df_base[c], errors='coerce')

summary = {
    'rows': int(len(df_base)),
    'countries': int(df_base['Country Name'].nunique()),
    'year_min': int(df_base['year'].min()),
    'year_max': int(df_base['year'].max()),
}

missing_report = (
    df_base[required_cols]
    .isna()
    .sum()
    .rename('missing_count')
    .to_frame()
)
missing_report['missing_share'] = missing_report['missing_count'] / len(df_base)

print('Data contract summary:')
print(summary)
print('\nMissingness report:')
display(missing_report)

EXPORT_TABLES['data_contract_missingness.csv'] = missing_report.reset_index().rename(columns={'index': 'column'})
log('Data contract check completed.')


## Method Availability Matrix

### Objective
Declare which Phase 1 branches are executable now versus scaffold-only.

### Caveat
A method can be technically runnable but still weak for identification if key inputs are missing.

### Interpretation Rule
- `Runnable`: can execute now with current files.
- `Scaffold`: section included, but notebook intentionally skips execution until required data exists.

### Next Data Needed
- Controls branch: `data/phase1_controls.csv`
- IV branch: `data/phase1_instruments.csv`


In [ ]:
availability = pd.DataFrame([
    {'method': 'Two-way FE baseline', 'status': 'Runnable', 'condition': 'Uses macro_growth_merged.csv'},
    {'method': 'Robustness pack (winsor + split + LOCO)', 'status': 'Runnable', 'condition': 'Uses macro_growth_merged.csv'},
    {'method': 'FE + controls', 'status': 'Runnable' if CONTROLS_PATH.exists() else 'Scaffold', 'condition': f'Needs {CONTROLS_PATH.name}'},
    {'method': 'IV / 2SLS', 'status': 'Runnable' if IV_PATH.exists() else 'Scaffold', 'condition': f'Needs {IV_PATH.name}'},
    {'method': 'Dynamic panel extension', 'status': 'Scaffold', 'condition': 'Template included; requires careful specification checks'},
    {'method': 'Forecast extension', 'status': 'Scaffold', 'condition': 'Template included; benchmark definitions required'},
])

display(availability)
EXPORT_TABLES['method_availability_matrix.csv'] = availability
log('Method availability matrix created.')


## Baseline FE Model: Inflation

### Objective
Estimate a two-way FE relationship between money growth and inflation.

### Identification Caveat
Two-way FE controls for time-invariant country effects and common year shocks, but does **not** resolve all endogeneity concerns.

### Interpretation Rule
Focus on coefficient sign, magnitude, and uncertainty for `m2_growth`, not just fit statistics.

### Next Data Needed
To raise causal credibility, add either controls with strong economic rationale or an explicit IV design.


In [ ]:
try:
    fe_inflation, panel_inflation = fit_twfe(df_base, outcome='inflation', regressor='m2_growth')
    tbl_fe_inflation = result_to_table(fe_inflation, model_name='fe_baseline', outcome='inflation')
    display(tbl_fe_inflation)

    FE_RESULTS['baseline_inflation'] = fe_inflation
    EXPORT_TABLES['fe_baseline_inflation.csv'] = tbl_fe_inflation

    coef = float(fe_inflation.params['m2_growth'])
    pval = float(fe_inflation.pvalues['m2_growth'])
    log(f'Baseline FE (inflation): coef={coef:.4f}, p={pval:.4g}, nobs={int(fe_inflation.nobs)}')
except Exception as e:
    log(f'Baseline FE (inflation) failed: {e}')


## Baseline FE Model: GDP Growth (Neutrality Check)

### Objective
Estimate whether money growth is associated with real GDP growth under two-way FE.

### Identification Caveat
A near-zero coefficient can still mask heterogeneous effects or dynamic channels.

### Interpretation Rule
Treat this as a baseline neutrality diagnostic, not a final structural conclusion.

### Next Data Needed
If this relationship becomes central, test heterogeneity by regimes and dynamic lags.


In [ ]:
try:
    fe_gdp, panel_gdp = fit_twfe(df_base, outcome='gdp_growth', regressor='m2_growth')
    tbl_fe_gdp = result_to_table(fe_gdp, model_name='fe_baseline', outcome='gdp_growth')
    display(tbl_fe_gdp)

    FE_RESULTS['baseline_gdp_growth'] = fe_gdp
    EXPORT_TABLES['fe_baseline_gdp_growth.csv'] = tbl_fe_gdp

    coef = float(fe_gdp.params['m2_growth'])
    pval = float(fe_gdp.pvalues['m2_growth'])
    log(f'Baseline FE (gdp_growth): coef={coef:.4f}, p={pval:.4g}, nobs={int(fe_gdp.nobs)}')
except Exception as e:
    log(f'Baseline FE (gdp_growth) failed: {e}')


## Robustness 1: Winsorization

### Objective
Check sensitivity to extreme observations by winsorizing tails.

### Identification Caveat
Winsorization addresses influence of outliers, not omitted variable bias.

### Interpretation Rule
If signs and rough magnitudes are stable relative to baseline FE, robustness improves.

### Next Data Needed
If instability appears, inspect outlier countries and measurement quality before redesigning the model.


In [ ]:
df_win = df_base.copy()
for c in ['m2_growth', 'inflation', 'gdp_growth']:
    df_win[c] = winsorize_series(df_win[c], lower_q=WINSOR_LOWER, upper_q=WINSOR_UPPER)

rows = []

for outcome in ['inflation', 'gdp_growth']:
    try:
        res, _ = fit_twfe(df_win, outcome=outcome, regressor='m2_growth')
        tmp = result_to_table(res, model_name='fe_winsorized', outcome=outcome)
        rows.append(tmp)
        log(f'Winsorized FE completed for {outcome}.')
    except Exception as e:
        log(f'Winsorized FE failed for {outcome}: {e}')

if rows:
    tbl_winsor = pd.concat(rows, ignore_index=True)
    display(tbl_winsor)
    EXPORT_TABLES['fe_winsorized_results.csv'] = tbl_winsor


## Robustness 2: Subsample by Period

### Objective
Test coefficient stability across time subsamples.

### Identification Caveat
Period splits reduce sample size and may reduce precision.

### Interpretation Rule
Large sign flips or severe magnitude instability indicate fragility.

### Next Data Needed
If instability is strong, add structural break logic or regime-based modeling.


In [ ]:
cut_year = int(df_base['year'].median())
period_slices = {
    f'pre_{cut_year}': df_base[df_base['year'] <= cut_year].copy(),
    f'post_{cut_year}': df_base[df_base['year'] > cut_year].copy(),
}

period_rows = []
for period_name, dfi in period_slices.items():
    for outcome in ['inflation', 'gdp_growth']:
        try:
            res, _ = fit_twfe(dfi, outcome=outcome, regressor='m2_growth')
            row = {
                'period': period_name,
                'outcome': outcome,
                'coef_m2_growth': float(res.params['m2_growth']),
                'p_value_m2_growth': float(res.pvalues['m2_growth']),
                'nobs': int(res.nobs),
                'r2_within': float(res.rsquared_within),
            }
            period_rows.append(row)
        except Exception as e:
            log(f'Period FE failed for {period_name}, {outcome}: {e}')

if period_rows:
    tbl_period = pd.DataFrame(period_rows)
    display(tbl_period)
    EXPORT_TABLES['fe_period_split_results.csv'] = tbl_period
    log(f'Period split robustness completed with cutoff year={cut_year}.')


## Robustness 3: Leave-One-Country-Out (LOCO)

### Objective
Measure whether the inflation FE result is overly driven by specific countries.

### Identification Caveat
LOCO diagnoses influence, not causal validity.

### Interpretation Rule
A tight LOCO coefficient distribution around baseline suggests stronger stability.

### Next Data Needed
If LOCO dispersion is wide, investigate country-level data quality and regime heterogeneity.


In [ ]:
if 'baseline_inflation' not in FE_RESULTS:
    log('LOCO skipped: baseline inflation FE result not available.')
else:
    all_countries = sorted(df_base['Country Name'].dropna().unique().tolist())
    if FAST_MODE and LOCO_MAX_COUNTRIES is not None:
        countries_to_test = all_countries[:LOCO_MAX_COUNTRIES]
    else:
        countries_to_test = all_countries

    baseline_coef = float(FE_RESULTS['baseline_inflation'].params['m2_growth'])
    loco_rows = []

    for i, ctry in enumerate(countries_to_test, start=1):
        dfi = df_base[df_base['Country Name'] != ctry].copy()
        try:
            res, _ = fit_twfe(dfi, outcome='inflation', regressor='m2_growth')
            loco_rows.append({
                'excluded_country': ctry,
                'coef_m2_growth': float(res.params['m2_growth']),
                'p_value_m2_growth': float(res.pvalues['m2_growth']),
                'nobs': int(res.nobs),
            })
        except Exception as e:
            log(f'LOCO failure for {ctry}: {e}')

    if loco_rows:
        tbl_loco = pd.DataFrame(loco_rows)
        tbl_loco['delta_vs_baseline'] = tbl_loco['coef_m2_growth'] - baseline_coef
        display(tbl_loco.head(10))

        summary_loco = tbl_loco['coef_m2_growth'].describe().to_frame('value')
        display(summary_loco)

        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(tbl_loco['coef_m2_growth'], bins=20, kde=True, ax=ax)
        ax.axvline(baseline_coef, color='red', linestyle='--', label='Baseline FE coef')
        ax.set_title('LOCO Distribution: FE Coefficient on m2_growth (Inflation Model)')
        ax.set_xlabel('Coefficient value')
        ax.legend()
        plt.show()

        EXPORT_TABLES['fe_loco_inflation.csv'] = tbl_loco
        EXPORT_TABLES['fe_loco_inflation_summary.csv'] = summary_loco.reset_index().rename(columns={'index': 'stat'})
        EXPORT_FIGS['loco_inflation_coef_distribution.png'] = fig

        log(f'LOCO completed for {len(countries_to_test)} countries (FAST_MODE={FAST_MODE}).')


## IV / 2SLS Section (Scaffold)

### Objective
Define an implementable IV branch with mandatory diagnostics.

### Identification Caveat
IV quality depends on relevance and exclusion. Weak instruments invalidate standard inference.

### Interpretation Rule
Proceed only if first-stage strength and validity checks are defensible.

### Next Data Needed
`data/phase1_instruments.csv` with at minimum:
- `Country Name`
- `year`
- `instrument_m2` (or clearly documented alternative instrument variable)


In [ ]:
RUN_IV_IF_AVAILABLE = False

if not IV_PATH.exists():
    log(f'IV scaffold not executed: file missing -> {IV_PATH}')
else:
    iv_df = pd.read_csv(IV_PATH)
    required_iv_cols = ['Country Name', 'year', 'instrument_m2']
    missing_iv_cols = [c for c in required_iv_cols if c not in iv_df.columns]

    if missing_iv_cols:
        log(f'IV scaffold not executed: missing columns -> {missing_iv_cols}')
    elif not RUN_IV_IF_AVAILABLE:
        log('IV file detected, but execution is disabled by default (RUN_IV_IF_AVAILABLE=False).')
        print('Template: merge iv_df with df_base on [Country Name, year], then run first-stage and IV2SLS with diagnostics.')
    else:
        # Minimal executable template if user toggles RUN_IV_IF_AVAILABLE=True.
        iv_merged = df_base.merge(iv_df[required_iv_cols], on=['Country Name', 'year'], how='inner').dropna()
        if len(iv_merged) < 200:
            log('IV execution stopped: too few observations after merge for stable diagnostics.')
        else:
            first_stage = sm.OLS(iv_merged['m2_growth'], sm.add_constant(iv_merged[['instrument_m2']])).fit()
            first_stage_f = float(first_stage.fvalue) if first_stage.fvalue is not None else np.nan
            log(f'IV first-stage F-stat (simple check) = {first_stage_f:.4f}')

            # Note: For full panel-IV with fixed effects, refine specification before treating output as final.
            iv_model = IV2SLS.from_formula('inflation ~ 1 + [m2_growth ~ instrument_m2]', data=iv_merged).fit(cov_type='robust')
            iv_tbl = pd.DataFrame({
                'term': iv_model.params.index,
                'coef': iv_model.params.values,
                'std_error': iv_model.std_errors.values,
                'p_value': iv_model.pvalues.values,
            })
            display(iv_tbl)
            EXPORT_TABLES['iv2sls_template_results.csv'] = iv_tbl
            log('IV template executed. Treat as provisional until full FE-IV specification is validated.')


## Dynamic Panel Section (Scaffold)

### Objective
Provide a dynamic specification template (lagged dependent variable + FE structure).

### Identification Caveat
Dynamic panel models need careful treatment of Nickell bias, instrument strategy, and lag structure.

### Interpretation Rule
Use this section for exploratory diagnostics, not final causal claims.

### Next Data Needed
Potentially richer panel history, stronger identification assumptions, and explicit dynamic-panel estimator choice.


In [ ]:
RUN_DYNAMIC = False

if not RUN_DYNAMIC:
    log('Dynamic panel scaffold not executed: set RUN_DYNAMIC=True after finalizing estimator assumptions.')
else:
    dyn = df_base.sort_values(['Country Name', 'year']).copy()
    dyn['inflation_l1'] = dyn.groupby('Country Name')['inflation'].shift(1)
    dyn = dyn.dropna(subset=['inflation', 'inflation_l1', 'm2_growth'])

    if len(dyn) < 500:
        log('Dynamic panel execution stopped: insufficient post-lag sample size.')
    else:
        dyn_panel = dyn.set_index(['Country Name', 'year']).sort_index()
        dyn_model = PanelOLS.from_formula(
            'inflation ~ 1 + inflation_l1 + m2_growth + EntityEffects + TimeEffects',
            data=dyn_panel,
        )
        dyn_res = dyn_model.fit(cov_type='clustered', cluster_entity=True)
        dyn_tbl = result_to_table(dyn_res, model_name='dynamic_fe_template', outcome='inflation')
        display(dyn_tbl)
        EXPORT_TABLES['dynamic_fe_template_results.csv'] = dyn_tbl
        log('Dynamic panel template executed. Validate estimator choice before interpretation.')


## Forecast Extension (Scaffold)

### Objective
Outline a practical predictive branch linking money growth to inflation forecasting.

### Identification Caveat
Forecast value does not imply causal identification.

### Interpretation Rule
Treat this as a secondary portfolio module after FE/robustness credibility is in place.

### Next Data Needed
Define benchmark models, rolling windows, and forecast evaluation protocol before production use.


In [ ]:
RUN_FORECAST = False

if not RUN_FORECAST:
    log('Forecast scaffold not executed: set RUN_FORECAST=True after choosing benchmark protocol.')
else:
    # Minimal rolling benchmark template.
    # Step 1: aggregate to global yearly means (placeholder design choice).
    yr = (
        df_base.groupby('year')[['inflation', 'm2_growth']]
        .mean()
        .dropna()
        .reset_index()
        .sort_values('year')
    )

    if len(yr) < 15:
        log('Forecast execution stopped: insufficient yearly observations for rolling validation.')
    else:
        split = int(len(yr) * 0.7)
        train = yr.iloc[:split]
        test = yr.iloc[split:]

        model = sm.OLS(train['inflation'], sm.add_constant(train[['m2_growth']])).fit()
        pred = model.predict(sm.add_constant(test[['m2_growth']], has_constant='add'))
        rmse = float(np.sqrt(np.mean((test['inflation'].values - pred.values) ** 2)))

        forecast_tbl = pd.DataFrame({
            'year': test['year'].values,
            'actual_inflation': test['inflation'].values,
            'pred_inflation': pred.values,
        })
        display(forecast_tbl.head())
        print(f'RMSE (placeholder benchmark): {rmse:.6f}')

        EXPORT_TABLES['forecast_template_predictions.csv'] = forecast_tbl
        EXPORT_TABLES['forecast_template_metrics.csv'] = pd.DataFrame([{'metric': 'rmse', 'value': rmse}])
        log('Forecast template executed. Upgrade design before using as portfolio final.')


## Portfolio Packaging Checklist

- [ ] One canonical notebook (`lucas_Phase 1.ipynb`) with deterministic `Run All` behavior.
- [ ] FE baseline + robustness tables exported to `outputs/phase1/tables/`.
- [ ] At least one stability diagnostic figure exported to `outputs/phase1/figures/`.
- [ ] Explicit caveats on identification quality.
- [ ] One interview-ready narrative linking method choice to business/policy relevance.

Target: high-signal, defensible output in <= 14 days.


In [ ]:
# Persist tables
written_tables = []
for filename, obj in EXPORT_TABLES.items():
    if not isinstance(obj, pd.DataFrame):
        continue
    out_path = TABLE_DIR / filename
    obj.to_csv(out_path, index=False)
    written_tables.append(str(out_path))

# Persist figures
written_figures = []
for filename, fig in EXPORT_FIGS.items():
    out_path = FIG_DIR / filename
    fig.savefig(out_path, dpi=160, bbox_inches='tight')
    written_figures.append(str(out_path))

# Persist log
log_path = OUTPUT_ROOT / 'phase1_model_log.md'
log_lines_fmt = ['# Phase 1 Model Log', ''] + [f'- {line}' for line in LOG_LINES]
log_path.write_text('\n'.join(log_lines_fmt))

# Persist export manifest
manifest = pd.DataFrame({
    'artifact_type': ['table'] * len(written_tables) + ['figure'] * len(written_figures) + ['log'],
    'path': written_tables + written_figures + [str(log_path)],
})
manifest_path = OUTPUT_ROOT / 'phase1_export_manifest.csv'
manifest.to_csv(manifest_path, index=False)

print('Export completed.')
print(f'Tables written: {len(written_tables)}')
print(f'Figures written: {len(written_figures)}')
print(f'Log path: {log_path}')
print(f'Manifest path: {manifest_path}')

if len(manifest) > 0:
    display(manifest)


## End-of-Notebook Decision Gate

### Continue
Continue this project if all are true:
1. FE baseline is interpretable and directionally coherent.
2. At least two robustness checks show acceptable stability.
3. You can explain identification limits and next upgrades clearly in interviews.

### Pause / Pivot
Pause or pivot if any are true:
1. Results are highly unstable under basic robustness.
2. IV/controls data construction dominates time without clear signal gain.
3. Work expands beyond the 14-day value cap.

### Immediate Next Upgrade Path
1. Add `data/phase1_controls.csv` and run FE + controls branch.
2. Add `data/phase1_instruments.csv` and run first-stage + IV diagnostics.
3. Finalize one short methods/results note from exported artifacts.
